# OP-FL v3 — 06d (version 3). Puissance du test verrouillé et amendement 1 de la spécification gelée

**Pourquoi.** La spécification gelée au 06c fixe comme critère principal C1 : R²_τ ≥ 0,25 **et** un gain en \|CITL\| moyen. Les hôpitaux verrouillés sont petits (100 à 499 séjours).
Le bruit d'estimation de chaque a_h y dépasse donc l'hétérogénéité réelle entre hôpitaux (version 1 de ce notebook : puissance du C1 gelé de 0,31).

**Corrections de la version 3** (relecture méthodologique du 27/10/2026) :
1. **Journal d'accès aux fichiers.** Chaque lecture (parquet, csv, npz, texte) est enregistrée, et les contrôles le démontrent : aucun fichier lu ne contient à la fois une ligne d'hôpital verrouillé et une colonne de label.
   Les descripteurs du développement sont relus depuis le 06b, sans rappeler `site_descriptors`, qui lit le manifest complet. La déclaration d'accès de l'amendement retrace l'historique.
2. **C4 est un test exact par permutation** (statistique du score), à la place du bootstrap percentile, dont l'erreur de type I simulée était de 0,047.
3. **Procédure graphique de multiplicité** (Bretz et al. 2009) : les conclusions de mortalité ne dépendent plus de C4.
4. **Simulations alignées** sur l'analyse finale (2000 tirages pour C2 et C4) ; 500 réplications de C4. Le risque global d'erreur de la procédure complète est vérifié sous les hypothèses nulles, avec des IC de Monte Carlo.
5. **Comparateurs et portée de b > 1/2 précisés** ; affirmation principale = transport du signal (C1a) ; limites de la simulation déclarées.

---

### Critères amendés (fixés ici, avant les résultats du test verrouillé)

Tous les tests sont **unilatéraux au seuil 2,5 %**.

**Estimand principal.** b dans a_h = α + b·(â_h − moyenne de â) + u_h + e_h, avec u_h ~ N(0, τ_r²) et e_h ~ N(0, v_h).
* a_h : CITL du modèle final `x_only` sur les patients `val + test` de l'hôpital verrouillé h.
* â_h : prédiction OP-Cal gelée, traitée comme une covariable fixe.
* v_h = 1 / Σ μ(1 − μ).

**Portée de b > 1/2.** Sous ce modèle et à moyenne correcte, l'erreur quadratique attendue de calibration est réduite de (2b − 1)·Var(â) par rapport à la **meilleure constante**. Cette constante est une référence théorique : la moyenne vraie des a_h verrouillés.
Cela ne garantit ni une baisse du \|CITL\| dans chaque hôpital, ni un gain de log-vraisemblance, ni la supériorité sur E1. C'est pourquoi C2 (contre E0) et les comparaisons à E1 sont distincts.

| Code | Hypothèse (H1) | Test | Poids initial |
|---|---|---|---|
| **C1a** | b > 0 : le signal se transporte (**affirmation principale**) | REML, Knapp-Hartung (t, k − 2 ddl) | 1/2 |
| **C4** | les hôpitaux qui dosent plus ont plus d'événements enregistrés que prévu (mécanisme Y* = D × Y) | score efficace U = Σ (z_h − z̄)(O_h − E_h) / (1 + E_h·τ²), τ² fixé sur le développement ; permutation de z entre hôpitaux, 2000 tirages (exact) | 1/2 |
| **C1b** | b > 0,5 : amélioration de la calibration | comme C1a | 0 |
| **C2** | Δℓ > 0 : gain de log-vraisemblance par patient contre E0 (aucune correction) | bootstrap de 2000 hôpitaux | 0 |

**Graphe de multiplicité** (Bonferroni pondéré, contrôle fort du risque global à 2,5 %) : C4 → C1a, C1a → C1b, C1b → C2, C2 → C4, chaque arête de poids 1.
* Si C4 est rejeté, C1a est testé à 2,5 %.
* Sinon, C1a, puis C1b, puis C2 sont testés à 1,25 %.
* Les conclusions de mortalité ne dépendent donc pas du lactate.

**Toujours rapportés.** α, b et R²_τ avec leurs IC ; bootstrap de b ; γ avec IC (bootstrap) et s'il contient 1 ; \|CITL\| et log-vraisemblance contre E0 et E1 ; EM et BBSE ; GRU-D ; S_mech_fixe ; C3 (k labels) en descriptif.

**Formulation des conclusions.**
* C1a : « le signal de calibration des sites se transporte ».
* C1a sans C1b : « … l'ampleur du gain reste incertaine sur ce jeu ».
* C1b : « amélioration confirmée par rapport à la meilleure constante, sous le modèle ».
* C2 : « gain de vraisemblance confirmé contre l'absence de correction ».
* C4 : « mécanisme du dosage confirmé ».
* Un échec est rapporté tel quel.

## Étape 0 — Journal d'accès, configuration, empreintes et statut de validation

Le journal d'accès est installé **avant toute lecture**.

`VALIDATION` reste à `False` tant que l'amendement n'a pas été relu par l'encadrante. Ensuite :
1. remplir `par` et `date` ;
2. mettre `valide` à `True` ;
3. ré-exécuter tout le notebook ;
4. déposer la version « validée ».

In [8]:
# ============================================================
# ÉTAPE 0 — JOURNAL D'ACCÈS, CONFIGURATION, EMPREINTES
# ============================================================
import sys, json, importlib, warnings, time, re, hashlib
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

# --- journal d'accès : chaque lecture de fichier est enregistrée (installé une seule fois par noyau)
LABEL_RE = re.compile(r"^(y|y_.*|aki_.*|hyper_.*|mortality|apache_pred.*|cr_peak|la_peak)$")
if "_AUDIT_ORIG" not in globals():
    _AUDIT_ORIG = {"read_parquet": pd.read_parquet, "read_csv": pd.read_csv, "load": np.load, "read_text": Path.read_text}
ACCESS_LOG = []
def _log(fn, path, obj=None, cols=None):
    if "site-packages" in str(path) or "dist-info" in str(path):     # lectures internes des bibliothèques
        return
    rec = {"fonction": fn, "fichier": str(path), "colonnes_demandees": None if cols is None else list(cols),
           "lignes": None, "hopitaux": set(), "colonnes_label": []}
    if isinstance(obj, pd.DataFrame):
        rec["lignes"] = len(obj)
        if "hospitalid" in obj.columns:
            rec["hopitaux"] = set(int(h) for h in pd.unique(obj["hospitalid"]))
        rec["colonnes_label"] = [str(c) for c in obj.columns if LABEL_RE.match(str(c))]
    ACCESS_LOG.append(rec)
def _rp(path, *a, **k):
    df = _AUDIT_ORIG["read_parquet"](path, *a, **k); _log("read_parquet", path, df, k.get("columns")); return df
def _rc(path, *a, **k):
    df = _AUDIT_ORIG["read_csv"](path, *a, **k); _log("read_csv", path, df, k.get("usecols")); return df
def _nl(path, *a, **k):
    _log("np.load", path); return _AUDIT_ORIG["load"](path, *a, **k)
def _rt(self, *a, **k):
    _log("read_text", self); return _AUDIT_ORIG["read_text"](self, *a, **k)
pd.read_parquet, pd.read_csv, np.load, Path.read_text = _rp, _rc, _nl, _rt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=RuntimeWarning)

VALIDATION = {"valide": True, "par": "Khalil (author)", "date": "27/10/2026"}       # à remplir après relecture par l'encadrante

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data_v1"
SRC_DIR = PROJECT_DIR / "src"
RES02 = PROJECT_DIR / "results" / "02_baselines"
RES04 = PROJECT_DIR / "results" / "04_observation_label"
RES05 = PROJECT_DIR / "results" / "05_raccourci_masquage"
RES06B = PROJECT_DIR / "results" / "06b_faisabilite_opcal"
RES06C = PROJECT_DIR / "results" / "06c_robustesse_opcal"
RES_DIR = PROJECT_DIR / "results" / "06d_amendement_puissance"
RES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR.resolve()))

EXPECTED_PROTOCOL_HASH = "d92099ea7c11d517"
protocol = json.loads((DATA_DIR / "splits" / "protocol.json").read_text(encoding="utf-8"))
assert protocol["hash"] == EXPECTED_PROTOCOL_HASH, f"Protocole inattendu ({protocol['hash']})."
FOLDS = sorted(int(k) for k in protocol["folds"])
FOLD_OF = {int(h): int(k) for k in protocol["folds"] for h in protocol["folds"][k]}
DEV = sorted(FOLD_OF)
LOCKED = set(int(h) for h in protocol["locked_test_hospitals"])
STAY_LABELS = RES04 / "stay_labels.parquet"

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

SPEC_PATH = RES06C / "specification_gelee.json"
assert SPEC_PATH.exists(), "Exécuter d'abord le notebook 06c."
SPEC = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
assert SPEC["statut"] == "gelée", f"Spécification au statut « {SPEC['statut']} » : pas d'amendement possible, revoir le 06c."
SPEC_SHA = sha256(SPEC_PATH)
m = re.search(r"SHA-256 `([0-9a-f]{16})", (RES06C / "rapport_06c.md").read_text(encoding="utf-8"))
SPEC_SHA_06C = m.group(1) if m else None
MOD_SHA = {f: sha256(SRC_DIR / f) for f in ("opcal_feas.py", "opcal_robust.py")}

N_REP_M, N_REP_G = 500, 500            # réplications par scénario (mortalité ; lactate)
N_PART, N_BOOT, N_PERM, SEED = 50, 2000, 2000, 2026   # N_BOOT et N_PERM : identiques à l'analyse finale
print(f"Protocole {protocol['hash']} | spécification « {SPEC['statut']} » du {SPEC['date']} | SHA-256 {SPEC_SHA[:16]}… "
      f"(06c : {SPEC_SHA_06C}…)")
print("Modules :", {k: v[:12] for k, v in MOD_SHA.items()}, "| validation :", VALIDATION)

Protocole d92099ea7c11d517 | spécification « gelée » du 2026-10-26T18:32:44 | SHA-256 302a9b448a1adcf4… (06c : 302a9b448a1adcf4…)
Modules : {'opcal_feas.py': '9770a8614e75', 'opcal_robust.py': 'eaa1313c3521'} | validation : {'valide': True, 'par': 'Khalil (author)', 'date': '27/10/2026'}


## Étape 1 — Module `src/opcal_amend.py`

La procédure confirmatoire et la simulation de puissance, qui appelle les mêmes fonctions avec les mêmes nombres de tirages. Le notebook 07 réutilisera ce module tel quel.
Il contient : C1a et C1b (REML + Knapp-Hartung), C2 (bootstrap des hôpitaux), C4 (permutation exacte) et le graphe de Bretz.

In [9]:
# %%writefile ../src/opcal_amend.py
# """
# opcal_amend — procédure confirmatoire de l'amendement 1 et sa puissance (OP-Cal).

# Généré par notebooks/06d_amendement_puissance.ipynb (étape 1) : modifier le notebook, pas ce fichier.
# Les fonctions de test sont celles que le notebook 07 appliquera au test verrouillé ; la simulation de
# puissance les appelle telles quelles, pour que puissance et analyse finale suivent la même procédure.

# Tous les tests sont unilatéraux au seuil 2,5 % (équivalent à un IC bilatéral à 95 %).
#   C1a : b > 0      (le signal de calibration des sites se transporte) — affirmation principale
#   C1b : b > 0,5    (sous le modèle linéaire et à moyenne correcte, l'erreur quadratique de calibration baisse de
#                     (2b − 1)·Var(â) par rapport à la meilleure constante, qui est une référence théorique)
#   C2  : Δℓ > 0     (gain de log-vraisemblance par patient d'OP-Cal contre E0, aucune correction ; comparaison
#                     distincte de C1b : elle inclut aussi le décalage moyen)
#   C4  : U > 0      (mécanisme Y* = D × Y : test exact par permutation du score efficace pondéré du log du taux de dosage)
# Multiplicité : procédure graphique (Bretz et al. 2009), voir GRAPH_WEIGHTS et GRAPH_EDGES.
# """
# import numpy as np
# import pandas as pd
# from scipy import optimize, stats

# import opcal_feas as oc

# ALPHA = 0.025   # unilatéral


# # ----------------------------------------------------------------------------------------------
# # C1 : méta-régression REML de a_h sur la prédiction gelée, IC de Knapp-Hartung
# # ----------------------------------------------------------------------------------------------

# def reml_metareg(y, x, v):
#     """a_h = alpha + b·x_h + u_h + e_h, u_h ~ N(0, tau2), e_h ~ N(0, v_h) ; x_h = â_h − moyenne(â).
#     tau2 par REML ; variance de b par Knapp-Hartung (sans troncature), t à k − 2 ddl."""
#     y, x, v = (np.asarray(a, float) for a in (y, x, v))
#     X = np.column_stack([np.ones(len(y)), x])

#     def fit(t2):
#         w = 1.0 / (v + t2)
#         A = X.T @ (X * w[:, None])
#         beta = np.linalg.solve(A, (X * w[:, None]).T @ y)
#         return w, A, beta, y - X @ beta

#     def nll(lt2):
#         t2 = np.exp(lt2)
#         w, A, beta, r = fit(t2)
#         return 0.5 * (np.sum(np.log(v + t2)) + np.linalg.slogdet(A)[1] + np.sum(w * r ** 2))

#     lt2 = optimize.minimize_scalar(nll, bounds=(np.log(1e-8), np.log(10.0)), method="bounded",
#                                    options={"xatol": 1e-6}).x
#     t2 = float(np.exp(lt2))
#     w, A, beta, r = fit(t2)
#     df = len(y) - 2
#     s2 = float(np.sum(w * r ** 2) / df)
#     se = float(np.sqrt(s2 * np.linalg.inv(A)[1, 1]))
#     return {"alpha": float(beta[0]), "b": float(beta[1]), "se_b": se, "df": df, "tau2_resid": t2, "k": len(y)}


# def p_slope(res, b0):
#     """p unilatéral pour H0 : b ≤ b0 (Knapp-Hartung)."""
#     return float(stats.t.sf((res["b"] - b0) / res["se_b"], res["df"]))


# def ci_slope(res, level=0.95):
#     q = stats.t.ppf(0.5 + level / 2, res["df"])
#     return res["b"] - q * res["se_b"], res["b"] + q * res["se_b"]


# def boot_slope(y, x, v, n_boot=2000, seed=0):
#     """Sensibilité : bootstrap des hôpitaux, tau2 réestimé à chaque tirage."""
#     y, x, v = (np.asarray(a, float) for a in (y, x, v))
#     rng = np.random.default_rng(seed)
#     out = np.empty(n_boot)
#     for i in range(n_boot):
#         j = rng.integers(0, len(y), len(y))
#         out[i] = reml_metareg(y[j], x[j] - x[j].mean(), v[j])["b"]
#     return out


# def c1_analysis(cal, ahat):
#     """cal : citl, var_citl par hôpital ; ahat : prédiction gelée (même index)."""
#     a = ahat.reindex(cal.index).to_numpy(float)
#     res = reml_metareg(cal["citl"].to_numpy(), a - a.mean(), cal["var_citl"].to_numpy())
#     lo, hi = ci_slope(res)
#     raw = oc.random_effects(cal["citl"], cal["var_citl"])
#     resid = oc.random_effects(cal["citl"].to_numpy() - a, cal["var_citl"].to_numpy())
#     return {**res, "b_IC_bas": lo, "b_IC_haut": hi, "p_C1a": p_slope(res, 0.0), "p_C1b": p_slope(res, 0.5),
#             "var_ahat": float(np.var(a)), "R2_tau": float(1 - resid["tau2"] / raw["tau2"]) if raw["tau2"] > 0 else np.nan}


# # ----------------------------------------------------------------------------------------------
# # C2 : gain de log-vraisemblance par patient (OP-Cal contre E0), bootstrap des hôpitaux
# # ----------------------------------------------------------------------------------------------

# def loglik(lp, y):
#     p = np.clip(oc.sigmoid(lp), 1e-12, 1 - 1e-12)
#     return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))


# def loglik_gain_by_hospital(stays, ahat, base=None):
#     """Δℓ_h = ℓ(lp + â_h) − ℓ(lp + base_h) sur les patients d'évaluation ; base = 0 (E0) par défaut."""
#     rows = []
#     for h, g in stays.groupby("hospitalid"):
#         lp, y = g["lp"].to_numpy(), g["y"].to_numpy(float)
#         b0 = 0.0 if base is None else float(base.loc[h])
#         rows.append({"hospitalid": h, "gain": loglik(lp + float(ahat.loc[h]), y) - loglik(lp + b0, y), "n": len(g)})
#     return pd.DataFrame(rows).set_index("hospitalid")


# def ratio_boot(num, den, n_boot=2000, seed=0):
#     """Σ num / Σ den, IC percentile et p unilatéral (H0 : ≤ 0) par bootstrap des hôpitaux."""
#     num, den = np.asarray(num, float), np.asarray(den, float)
#     rng = np.random.default_rng(seed)
#     idx = rng.integers(0, len(num), (n_boot, len(num)))
#     bs = num[idx].sum(1) / den[idx].sum(1)
#     est = float(num.sum() / den.sum())
#     return est, float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975)), float((1 + np.sum(bs <= 0)) / (n_boot + 1))


# # ----------------------------------------------------------------------------------------------
# # C4 : test exact par permutation (statistique du score) ; γ estimé à part (descriptif)
# # ----------------------------------------------------------------------------------------------

# def _o_minus_e(stays, z_by_h):
#     """O_h − E_h et E_h par hôpital sous le modèle nul logit P(Y* = 1) = lp + alpha (alpha commun, MLE avec offset)."""
#     s = stays[stays["hospitalid"].isin(z_by_h.dropna().index)]
#     lp, y = s["lp"].to_numpy(), s["y"].to_numpy(float)
#     alpha = oc.offset_fit(lp, y)[0]
#     mu = oc.sigmoid(lp + alpha)
#     g = s["hospitalid"].to_numpy()
#     oe = pd.Series(y - mu).groupby(g).sum()
#     E = pd.Series(mu).groupby(g).sum().reindex(oe.index)
#     return oe, E, z_by_h.reindex(oe.index).to_numpy(float)


# def gamma_score_test(stays, z_by_h, n_perm=2000, seed=0, tau2=0.0):
#     """C4 : U = Σ_h (z_h − z̄)·(O_h − E_h) / (1 + E_h·tau2). « Les hôpitaux qui dosent plus ont plus d'événements enregistrés
#     que prévu. » Le poids 1 / (1 + E_h·tau2) est celui du score efficace quand les hôpitaux sont surdispersés
#     (Var(O_h) ≈ E_h·(1 + E_h·tau2)) ; tau2 est fixé à l'avance (développement). tau2 = 0 : score non pondéré.
#     p = (1 + #{U* ≥ U}) / (B + 1), z permuté entre hôpitaux : exact si z est indépendant des écarts O − E sous H0
#     (les poids ne dépendent que du modèle nul, donc pas de la permutation)."""
#     oe, E, z = _o_minus_e(stays, z_by_h)
#     R = (oe / (1.0 + E * tau2)).to_numpy()
#     zc = z - z.mean()
#     U = float(zc @ R)
#     rng = np.random.default_rng(seed)
#     perms = np.argsort(rng.random((n_perm, len(zc))), axis=1)
#     Ustar = zc[perms] @ R
#     return {"U": U, "p_C4": float((1 + np.sum(Ustar >= U)) / (n_perm + 1)), "hopitaux": len(zc),
#             "evenements_attendus": float(E.sum()), "information": float(np.sum(zc ** 2 * E / (1.0 + E * tau2)))}


# def c4_power_from_dev(p_dev, info_dev, E_locked, var_z, tau2, alphas=(ALPHA / 2, ALPHA)):
#     """Puissance de C4 extrapolée du développement : z observé au développement, mis à l'échelle de l'information
#     attendue au test verrouillé (Σ Var(z)·E_h / (1 + E_h·tau2)). Approximation : traite le z du développement comme vrai."""
#     z_dev = float(stats.norm.isf(p_dev))
#     info_locked = float(var_z * np.sum(E_locked / (1.0 + E_locked * tau2)))
#     z_lock = z_dev * np.sqrt(info_locked / info_dev)
#     return {"z_dev": z_dev, "z_verrouille": z_lock, **{f"puissance_a_{a:.4f}": float(stats.norm.sf(stats.norm.isf(a) - z_lock)) for a in alphas}}


# def gamma_estimate(stays, z_by_h, n_boot=2000, seed=0):
#     """γ (logistique avec offset lp, z centré) et IC percentile par bootstrap des hôpitaux : estimation, pas test."""
#     s = stays[stays["hospitalid"].isin(z_by_h.dropna().index)]
#     z = s["hospitalid"].map(z_by_h).to_numpy(float) - np.mean(z_by_h.dropna())
#     g = s["hospitalid"].to_numpy()
#     lp, y = s["lp"].to_numpy(), s["y"].to_numpy(float)
#     est = oc.offset_slope_fit(lp, y, z)[1]
#     hs = np.unique(g)
#     rows = {h: np.flatnonzero(g == h) for h in hs}
#     rng = np.random.default_rng(seed)
#     bs = np.empty(n_boot)
#     for b in range(n_boot):
#         i = np.concatenate([rows[h] for h in rng.choice(hs, len(hs))])
#         bs[b] = oc.offset_slope_fit(lp[i], y[i], z[i])[1]
#     lo, hi = float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))
#     return {"gamma": float(est), "IC_bas": lo, "IC_haut": hi, "IC_contient_1": bool(lo <= 1 <= hi)}


# # ----------------------------------------------------------------------------------------------
# # Multiplicité : procédure graphique (Bretz, Maurer, Brannath et Posch 2009), unilatérale à 2,5 %
# #   poids initiaux : C1a 1/2, C4 1/2 ; arêtes C4 → C1a, C1a → C1b, C1b → C2, C2 → C4 (poids 1).
# #   Les conclusions de mortalité ne dépendent pas de C4 : si C4 échoue, C1a, C1b et C2 sont testés à 1,25 % ;
# #   si C4 est rejeté, son seuil passe à C1a (2,5 %). Contrôle fort du risque global d'erreur (Bonferroni pondéré).
# # ----------------------------------------------------------------------------------------------

# GRAPH_WEIGHTS = {"C1a": 0.5, "C4": 0.5, "C1b": 0.0, "C2": 0.0}
# GRAPH_EDGES = {("C4", "C1a"): 1.0, ("C1a", "C1b"): 1.0, ("C1b", "C2"): 1.0, ("C2", "C4"): 1.0}


# def graph_test(p, weights=GRAPH_WEIGHTS, edges=GRAPH_EDGES, alpha=ALPHA):
#     """Algorithme 1 de Bretz et al. (2009). p : dict hypothèse -> p unilatéral. Retourne dict hypothèse -> rejetée."""
#     H = list(weights)
#     w = dict(weights)
#     G = {(i, j): edges.get((i, j), 0.0) for i in H for j in H if i != j}
#     active, rej = set(H), {h: False for h in H}
#     while True:
#         cand = [h for h in active if w[h] > 0 and p[h] <= w[h] * alpha]
#         if not cand:
#             return rej
#         i = min(cand, key=lambda h: p[h])
#         rej[i] = True
#         active.discard(i)
#         for j in active:
#             w[j] += w[i] * G[(i, j)]
#         w[i] = 0.0
#         newG = {}
#         for j in active:
#             for k in active:
#                 if j != k:
#                     d = 1.0 - G[(j, i)] * G[(i, j)]
#                     newG[(j, k)] = (G[(j, k)] + G[(j, i)] * G[(i, k)]) / d if d > 1e-12 else 0.0
#         G.update(newG)


# # ----------------------------------------------------------------------------------------------
# # Simulation de puissance : mêmes fonctions et mêmes nombres de tirages que l'analyse finale
# # ----------------------------------------------------------------------------------------------

# def _sample_lp(pools, n, rng):
#     """lp d'un hôpital simulé : tirage avec remise dans les patients d'un hôpital de développement pris au hasard."""
#     pool = pools[rng.integers(0, len(pools))]
#     return pool[rng.integers(0, len(pool), n)]


# def simulate_mortality(n_eval, lp_pools, b, tau2, var_ahat, n_rep=500, n_boot=2000, seed=0, k_lab=25, n0_fixed=10.0):
#     """Réplications du test verrouillé : â_h ~ N(0, var_ahat) (dispersion des prédictions du développement),
#     a_h = b·â_h + u_h avec Var(a_h) = tau2 (si b²·var_ahat ≤ tau2). p de C1a, C1b (Knapp-Hartung), C2 (bootstrap
#     de n_boot hôpitaux, comme l'analyse finale), et puissance des C1 et C3 tels que gelés au 06c."""
#     rng = np.random.default_rng(seed)
#     K = len(n_eval)
#     s2u = max(tau2 - b ** 2 * var_ahat, 1e-4)
#     out = []
#     for r in range(n_rep):
#         ahat = rng.normal(0, np.sqrt(var_ahat), K)
#         a = b * ahat + rng.normal(0, np.sqrt(s2u), K)
#         c, v, gain = np.empty(K), np.empty(K), np.empty(K)
#         reg = np.full((K, 3), np.nan)      # C3 gelé : |CITL| déploiement (local, rétrécissement fixe, EB OP-Cal)
#         for h in range(K):
#             lp = _sample_lp(lp_pools, int(n_eval[h]), rng)
#             y = (rng.random(len(lp)) < oc.sigmoid(lp + a[h])).astype(float)
#             c[h], v[h] = oc.offset_fit(lp, y)
#             gain[h] = loglik(lp + ahat[h], y) - loglik(lp, y)
#             perm = rng.permutation(len(lp)); half = len(lp) // 2
#             if half >= k_lab:
#                 L, Dp = perm[:half][:k_lab], perm[half:]
#                 res_ = lambda off: abs(oc.offset_fit(lp[Dp] + off, y[Dp])[0])
#                 reg[h] = (res_(oc.offset_fit(lp[L], y[L])[0]),
#                           res_(oc.offset_fit(lp[L], y[L], prior_mean=0.0, prior_var=1.0 / n0_fixed)[0]),
#                           res_(oc.offset_fit(lp[L], y[L], prior_mean=ahat[h], prior_var=s2u + 1e-3)[0]))
#         res = reml_metareg(c, ahat - ahat.mean(), v)
#         _, _, _, p2 = ratio_boot(gain, n_eval, n_boot, int(rng.integers(1e9)))
#         t_raw = oc.random_effects(c, v)["tau2"]
#         r2 = 1 - oc.random_effects(c - ahat, v)["tau2"] / t_raw if t_raw > 0 else np.nan
#         diff = np.abs(c) - np.abs(c - ahat)          # C1 gelé : E1 = moyenne vraie (nulle dans la simulation)
#         bs = diff[rng.integers(0, K, (n_boot, K))].mean(1)
#         ok = np.isfinite(reg[:, 0])
#         comp = 0 if np.nanmean(reg[:, 0]) <= np.nanmean(reg[:, 1]) else 1
#         d3 = reg[ok, comp] - reg[ok, 2]
#         c3 = bool(ok.sum() > 2 and np.quantile(d3[rng.integers(0, ok.sum(), (n_boot, ok.sum()))].mean(1), 0.025) > 0)
#         out.append({"rep": r, "b_est": res["b"], "p_C1a": p_slope(res, 0.0), "p_C1b": p_slope(res, 0.5), "p_C2": p2, "C3_gele": c3,
#                     "C1_gele": bool(np.isfinite(r2) and r2 >= 0.25 and np.quantile(bs, 0.025) > 0), "R2_tau": r2,
#                     "largeur_IC": 2 * stats.t.ppf(0.975, res["df"]) * res["se_b"]})
#     return pd.DataFrame(out)


# def simulate_gamma(n_eval48, lp_pools, z_pool, gamma, tau2_resid, n_rep=500, n_perm=2000, seed=0, tau2_weight=0.0):
#     """Réplications de C4 : z_h tiré parmi les log taux de dosage du développement ; Y* ~ lp + γ·(z − z̄) + u_h.
#     Test exact par permutation (n_perm, comme l'analyse finale) ; γ estimé par MLE (sans bootstrap)."""
#     rng = np.random.default_rng(seed)
#     K = len(n_eval48)
#     out = []
#     for r in range(n_rep):
#         z = rng.choice(z_pool, K)
#         u = rng.normal(0, np.sqrt(max(tau2_resid, 1e-6)), K)
#         parts = []
#         for h in range(K):
#             lp = _sample_lp(lp_pools, int(n_eval48[h]), rng)
#             y = (rng.random(len(lp)) < oc.sigmoid(lp + gamma * (z[h] - z.mean()) + u[h])).astype(float)
#             parts.append(pd.DataFrame({"hospitalid": h, "lp": lp, "y": y}))
#         st = pd.concat(parts, ignore_index=True)
#         zs = pd.Series(z, index=range(K))
#         t = gamma_score_test(st, zs, n_perm, int(rng.integers(1e9)), tau2=tau2_weight)
#         g_est = oc.offset_slope_fit(st["lp"].to_numpy(), st["y"].to_numpy(float),
#                                     st["hospitalid"].map(zs).to_numpy(float) - z.mean())[1]
#         out.append({"rep": r, "gamma_est": g_est, "p_C4": t["p_C4"], "evenements": int(st["y"].sum())})
#     return pd.DataFrame(out)


# def mc_ci(k, n, level=0.95):
#     """IC exact (Clopper-Pearson) d'une proportion de Monte Carlo."""
#     a = (1 - level) / 2
#     lo = stats.beta.ppf(a, k, n - k + 1) if k > 0 else 0.0
#     hi = stats.beta.ppf(1 - a, k + 1, n - k) if k < n else 1.0
#     return float(lo), float(hi)


# def procedure_table(sim_m, sim_g, true_null=(), alpha=ALPHA, seed=0):
#     """Procédure complète (graphe) sur des réplications de mortalité appariées au hasard avec des réplications de C4
#     (indépendantes). Puissance de chaque hypothèse, puissance de chaque test seul, et proportion de réplications avec
#     au moins un rejet à tort parmi `true_null` (risque global d'erreur, IC de Clopper-Pearson)."""
#     rng = np.random.default_rng(seed)
#     pg = sim_g["p_C4"].to_numpy()[rng.integers(0, len(sim_g), len(sim_m))]
#     dec = [graph_test({"C1a": a, "C4": g, "C1b": b, "C2": c}, alpha=alpha)
#            for a, g, b, c in zip(sim_m["p_C1a"], pg, sim_m["p_C1b"], sim_m["p_C2"])]
#     n = len(dec)
#     out = {"C1_gele": float(np.mean(sim_m["C1_gele"])), "C3_gele": float(np.mean(sim_m["C3_gele"])),
#            "C1a_seul": float(np.mean(sim_m["p_C1a"] <= alpha)), "C1b_seul": float(np.mean(sim_m["p_C1b"] <= alpha)),
#            "C2_seul": float(np.mean(sim_m["p_C2"] <= alpha)), "C4_seul": float(np.mean(sim_g["p_C4"] <= alpha))}
#     for h in ("C1a", "C4", "C1b", "C2"):
#         out[f"{h}_confirme"] = float(np.mean([d[h] for d in dec]))
#     if true_null:
#         k = int(sum(any(d[h] for h in true_null) for d in dec))
#         out["erreur_globale"] = k / n
#         out["erreur_globale_IC_bas"], out["erreur_globale_IC_haut"] = mc_ci(k, n)
#     out.update({"b_estime_median": float(sim_m["b_est"].median()), "largeur_IC_mediane": float(sim_m["largeur_IC"].median()),
#                 "R2_tau_median": float(np.nanmedian(sim_m["R2_tau"]))})
#     return out

## Étape 2 — Cibles et prédictions gelées sur les hôpitaux de développement

Les cibles a_h sont les mêmes qu'aux 06b et 06c. Les descripteurs sont relus depuis `results/06b_faisabilite_opcal/descripteurs_site.csv` (hôpitaux de développement uniquement) : `site_descriptors`, qui lit le manifest complet, n'est pas rappelé.
Le contrôle de l'étape 6 vérifie qu'on retrouve le R²_τ de l'ablation du 06c avec les 44 descripteurs gelés.

In [10]:
# ============================================================
# ÉTAPE 2 — CIBLES ET PRÉDICTIONS GELÉES (DÉVELOPPEMENT)
# ============================================================
import opcal_feas as oc, opcal_robust as orb, opcal_amend as am
for m_ in (oc, orb, am):
    importlib.reload(m_)

TASKS = {"mortalite_x_only": {"pred_dir": RES02 / "preds", "name": "x_only", "label": "y_mortality", "pop": "tous"},
         "lactate_x_only": {"pred_dir": RES05 / "preds", "name": "A_lac_neg_x_only", "label": "hyper_neg", "pop": "present48"}}
st = pd.read_parquet(STAY_LABELS).set_index("stay_id")
assert set(st["hospitalid"]) <= set(DEV), "stay_labels.parquet contient des hôpitaux hors développement."
POP = {"tous": st.index, "present48": st.index[st["present48"].astype(bool)]}
CAL, STAYS = {}, {}
for t, c_ in TASKS.items():
    d, _ = oc.load_transfer(c_["pred_dir"], c_["name"], FOLDS)
    if c_["label"] != "y_mortality" or "y" not in d.columns:
        d = d.drop(columns="y", errors="ignore")
        d = d[d["stay_id"].isin(POP[c_["pop"]])].copy()
        d["y"] = d["stay_id"].map(st[c_["label"]]).astype(float)
    d = d[d["y"].notna()].reset_index(drop=True)
    CAL[t], STAYS[t] = oc.hospital_calibration(d), d

D_SITE = pd.read_csv(RES06B / "descripteurs_site.csv", index_col=0)
D_SITE.index = D_SITE.index.astype(int)
assert set(D_SITE.index) == set(DEV), "Descripteurs du 06b : hôpitaux inattendus."
D_M = D_SITE.join(CAL["mortalite_x_only"][["mean_lp", "sd_lp"]])
FROZEN = SPEC["meta_modele"]["descripteurs"]
assert all(c in D_M.columns for c in FROZEN), "Descripteurs gelés introuvables."
cal = CAL["mortalite_x_only"]
PRIOR_DEV = oc.crossfit_prior(cal, D_M, FROZEN)
ev = oc.evaluate_prior(cal, PRIOR_DEV)
print(f"Mortalité x_only, {len(FROZEN)} descripteurs gelés, plis du protocole : R²_τ = {ev['R2_tau']:.3f} ; |CITL| = {ev['mean_abs_citl']:.3f}")

Mortalité x_only, 44 descripteurs gelés, plis du protocole : R²_τ = 0.608 ; |CITL| = 0.244


## Étape 3 — Pente de calibration au niveau des sites, sur le développement

La pente est mesurée avec les fonctions de C1a et C1b, sur les prédictions croisées : plis du protocole, 50 partitions aléatoires et leave-one-hospital-out.
On mesure aussi le gain de log-vraisemblance contre E0 et contre E1. Ce sont des estimations descriptives.

In [11]:
# ============================================================
# ÉTAPE 3 — PENTE b AU NIVEAU DES SITES (DÉVELOPPEMENT)
# ============================================================
rows = []
def slope_row(name, prior):
    r = am.c1_analysis(cal, prior["a_hat"])
    rows.append({"schéma": name, "b": r["b"], "b_IC_bas": r["b_IC_bas"], "b_IC_haut": r["b_IC_haut"], "alpha": r["alpha"],
                 "p_C1a": r["p_C1a"], "p_C1b": r["p_C1b"], "R2_tau": r["R2_tau"], "var_ahat": r["var_ahat"]})
    return r
r_proto = slope_row("plis du protocole", PRIOR_DEV)
rng = np.random.default_rng(SEED)
part = []
for i in range(N_PART):
    f = np.empty(len(cal), int)
    f[rng.permutation(len(cal))] = np.arange(len(cal)) % len(FOLDS)
    c2 = cal.copy(); c2["fold"] = f
    part.append(am.c1_analysis(cal, oc.crossfit_prior(c2, D_M, FROZEN)["a_hat"]))
PART = pd.DataFrame(part)
_, pr_loho = orb.leave_one_hospital_out(cal, D_M, FROZEN)
slope_row("leave-one-hospital-out", pr_loho)
rows.append({"schéma": f"{N_PART} partitions (médiane)", "b": PART["b"].median(), "b_IC_bas": PART["b"].quantile(0.05),
             "b_IC_haut": PART["b"].quantile(0.95), "alpha": PART["alpha"].median(), "p_C1a": PART["p_C1a"].median(),
             "p_C1b": PART["p_C1b"].median(), "R2_tau": PART["R2_tau"].median(), "var_ahat": PART["var_ahat"].median()})
TAB_DEV = pd.DataFrame(rows).set_index("schéma")
TAB_DEV.to_csv(RES_DIR / "pente_developpement.csv")
print("Pente de calibration des sites (développement) ; pour les partitions, les bornes sont les 5e et 95e centiles :")
display(TAB_DEV.round(3))

bs = am.boot_slope(cal["citl"], PRIOR_DEV["a_hat"].reindex(cal.index) - PRIOR_DEV["a_hat"].mean(), cal["var_citl"], N_BOOT, SEED)
print(f"Bootstrap des hôpitaux (plis du protocole) : b = {r_proto['b']:.3f} [{np.quantile(bs, 0.025):.3f} ; {np.quantile(bs, 0.975):.3f}]")

E1_DEV = oc.crossfit_prior(cal, D_M, [])["a_hat"]
g0 = am.loglik_gain_by_hospital(STAYS["mortalite_x_only"], PRIOR_DEV["a_hat"])
g1 = am.loglik_gain_by_hospital(STAYS["mortalite_x_only"], PRIOR_DEV["a_hat"], base=E1_DEV)
GAIN_DEV = pd.DataFrame([dict(zip(["gain_par_patient", "IC_bas", "IC_haut", "p_unilateral"], am.ratio_boot(g["gain"], g["n"], N_BOOT, SEED)), contre=k)
                         for k, g in (("E0", g0), ("E1", g1))]).set_index("contre")
print("Gain de log-vraisemblance par patient (développement, prédictions croisées) :")
display(GAIN_DEV.round(5))

B_DEV, VAR_AHAT, TAU2_DEV = float(PART["b"].median()), float(PART["var_ahat"].median()), oc.random_effects(cal["citl"], cal["var_citl"])["tau2"]
print(f"Paramètres retenus pour la simulation : b = {B_DEV:.3f} (médiane des partitions) ; Var(â) = {VAR_AHAT:.4f} ; τ² brut = {TAU2_DEV:.4f}")

Pente de calibration des sites (développement) ; pour les partitions, les bornes sont les 5e et 95e centiles :


,b,b_IC_bas,b_IC_haut,alpha,p_C1a,p_C1b,R2_tau,var_ahat
schéma,,,,,,,,
plis du protocole,1.162,0.772,1.552,0.055,0.0,0.001,0.608,0.042
leave-one-hospital-out,0.968,0.628,1.309,0.057,0.0,0.004,0.556,0.057
50 partitions (médiane),0.916,0.756,1.085,0.058,0.0,0.024,0.399,0.047


Bootstrap des hôpitaux (plis du protocole) : b = 1.162 [0.824 ; 1.465]
Gain de log-vraisemblance par patient (développement, prédictions croisées) :


,gain_par_patient,IC_bas,IC_haut,p_unilateral
contre,,,,
E0,0.00162,0.00054,0.00286,0.0010
E1,0.00154,0.00060,0.00271,0.0005


Paramètres retenus pour la simulation : b = 0.916 (médiane des partitions) ; Var(â) = 0.0470 ; τ² brut = 0.0812


## Étape 4 — Puissance et risque global d'erreur, sur les vrais effectifs du test verrouillé

Du manifest, seules les colonnes `hospitalid`, `role`, `n_val` et `n_test` sont lues.

**Hypothèses de simulation (déclarées comme limites).**
* Les lp sont tirés, sans leurs labels, parmi les patients d'évaluation d'un hôpital de développement pris au hasard.
* La part des séjours présents à 48 h est celle du développement.
* â_h a la dispersion du développement.
* Le taux de dosage z_h est tiré parmi ceux du développement.

C'est une analyse de sensibilité sous ces hypothèses, pas une garantie de puissance.

**Risque global d'erreur.** La procédure complète (le graphe) est appliquée sous quatre configurations nulles :
* nul global ;
* b = 0 avec un γ réel ;
* b = 0,5 avec un γ réel ;
* γ = 0 avec un b réel.

La proportion de réplications avec au moins un rejet à tort est donnée avec un IC exact (Clopper-Pearson).

In [12]:
# ============================================================
# ÉTAPE 4 — PUISSANCE ET RISQUE GLOBAL D'ERREUR
# ============================================================
man_cols = ["hospitalid", "role", "n_val", "n_test"]
man = pd.read_csv(DATA_DIR / "clients" / "manifest.csv", usecols=man_cols)
man_locked = man[man["hospitalid"].isin(LOCKED)]
N_EVAL = (man_locked["n_val"] + man_locked["n_test"]).to_numpy()
ev_dev = st[st["split"].isin(["val", "test"])]
SHARE48 = float(ev_dev["present48"].astype(bool).mean())
N_EVAL48 = np.maximum((N_EVAL * SHARE48).round().astype(int), 5)
MORT_DEV = float(STAYS["mortalite_x_only"]["y"].mean())
EXP_DEATHS = float(np.median(N_EVAL) * MORT_DEV)
V_MED = 1.0 / (np.median(N_EVAL) * MORT_DEV * (1 - MORT_DEV))
K_CONF = int(SPEC["k_confirmatoire"])
print(f"Test verrouillé : {len(N_EVAL)} hôpitaux ; patients d'évaluation médiane {np.median(N_EVAL):.0f} [{N_EVAL.min()} ; {N_EVAL.max()}] ; "
      f"présents à 48 h médiane {np.median(N_EVAL48):.0f} ; décès attendus par hôpital ≈ {EXP_DEATHS:.1f} ; v_h médiane ≈ {V_MED:.3f}")

LP_M = [g["lp"].to_numpy() for _, g in STAYS["mortalite_x_only"].groupby("hospitalid")]
LP_L = [g["lp"].to_numpy() for _, g in STAYS["lactate_x_only"].groupby("hospitalid")]
Z_DEV = D_SITE["log_test_rate_lactate"]
Z_POOL = Z_DEV.dropna().to_numpy()
cal_l = CAL["lactate_x_only"]
pr_mech = oc.crossfit_prior(cal_l, D_SITE, [], fixed_beta=("log_test_rate_lactate", 1.0))
TAU2_MECH = oc.random_effects(cal_l["citl"].to_numpy() - pr_mech["a_hat"].reindex(cal_l.index).to_numpy(), cal_l["var_citl"].to_numpy())["tau2"]
G_DEV = am.gamma_estimate(STAYS["lactate_x_only"], Z_DEV, N_BOOT, SEED)
C4_DEV = am.gamma_score_test(STAYS["lactate_x_only"], Z_DEV, N_PERM, SEED, tau2=TAU2_MECH)      # statistique confirmatoire
C4_DEV_NP = am.gamma_score_test(STAYS["lactate_x_only"], Z_DEV, N_PERM, SEED, tau2=0.0)        # non pondérée (descriptif)
PREV_LAC = float(STAYS["lactate_x_only"]["y"].mean())
C4_EMP = am.c4_power_from_dev(C4_DEV["p_C4"], C4_DEV["information"], N_EVAL48 * PREV_LAC, float(np.var(Z_POOL)), TAU2_MECH)
print(f"Lactate (développement) : γ = {G_DEV['gamma']:.3f} [{G_DEV['IC_bas']:.3f} ; {G_DEV['IC_haut']:.3f}] ; τ² résiduel (γ = 1) = {TAU2_MECH:.3f}")
print(f"  test exact, score efficace (τ² = {TAU2_MECH:.3f}) : p = {C4_DEV['p_C4']:.4f} ; score non pondéré : p = {C4_DEV_NP['p_C4']:.4f}")
print(f"  puissance de C4 extrapolée du développement : {C4_EMP['puissance_a_0.0250']:.2f} à 2,5 % ; {C4_EMP['puissance_a_0.0125']:.2f} à 1,25 % "
      f"(z dév. {C4_EMP['z_dev']:.2f} → z verrouillé attendu {C4_EMP['z_verrouille']:.2f})")

t0 = time.time()
SIM_G = {name: am.simulate_gamma(N_EVAL48, LP_L, Z_POOL, g, TAU2_MECH, N_REP_G, N_PERM, SEED + i, tau2_weight=TAU2_MECH)
         for i, (name, g) in enumerate([("γ = 0", 0.0), ("γ = 0,5", 0.5), ("γ du dév.", G_DEV["gamma"])])}
print(f"Simulations de C4 : {time.time() - t0:.0f} s")

SCEN = {"b = 0": (0.0, TAU2_DEV), "b = 0,5": (0.5, TAU2_DEV), "b = 0,8": (0.8, TAU2_DEV), "b = 1": (1.0, TAU2_DEV),
        "b du dév.": (B_DEV, TAU2_DEV), "b = 0,8 ; τ² = 0,12": (0.8, 0.12), "b = 1 ; τ² = 0,12": (1.0, 0.12),
        "b du dév. ; τ² = 0,12": (B_DEV, 0.12)}
t0 = time.time()
SIM_M = {k: am.simulate_mortality(N_EVAL, LP_M, b, tau2, VAR_AHAT, N_REP_M, N_BOOT, SEED + 100 + i, k_lab=K_CONF)
         for i, (k, (b, tau2)) in enumerate(SCEN.items())}
print(f"Simulations de mortalité : {time.time() - t0:.0f} s ({N_REP_M} réplications par scénario)")
C1A_FALLBACK = float(np.mean(SIM_M["b du dév."]["p_C1a"] <= am.ALPHA / 2))
print(f"Puissance de C1a si C4 n'est pas rejeté (seuil 1,25 %), b du dév. : {C1A_FALLBACK:.3f}")

rows = []
for i, k in enumerate(["b = 0,8", "b = 1", "b du dév.", "b = 0,8 ; τ² = 0,12", "b = 1 ; τ² = 0,12", "b du dév. ; τ² = 0,12"]):
    b, tau2 = SCEN[k]
    rows.append({"scénario": k, "b": b, "tau2": tau2, "contrainte_tau2": b ** 2 * VAR_AHAT > tau2,
                 **am.procedure_table(SIM_M[k], SIM_G["γ du dév."], seed=SEED + i)})
rows.append({"scénario": "b du dév. ; γ = 0,5", "b": B_DEV, "tau2": TAU2_DEV, "contrainte_tau2": False,
             **am.procedure_table(SIM_M["b du dév."], SIM_G["γ = 0,5"], seed=SEED + 50)})
TAB_POW = pd.DataFrame(rows).set_index("scénario")
TAB_POW.to_csv(RES_DIR / "puissance.csv")
pow_cols = ["b", "tau2", "C1_gele", "C3_gele", "C1a_confirme", "C4_confirme", "C1b_confirme", "C2_confirme",
            "C1a_seul", "C1b_seul", "C2_seul", "C4_seul", "b_estime_median", "largeur_IC_mediane"]
display(TAB_POW[pow_cols].round(3))

NULLS = [("nul global : b = 0, γ = 0", "b = 0", "γ = 0", ("C1a", "C1b", "C2", "C4")),
         ("b = 0 ; γ du dév.", "b = 0", "γ du dév.", ("C1a", "C1b", "C2")),
         ("b = 0,5 ; γ du dév.", "b = 0,5", "γ du dév.", ("C1b", "C2")),
         ("b du dév. ; γ = 0", "b du dév.", "γ = 0", ("C4",))]
TAB_FWER = pd.DataFrame([{"configuration nulle": name, "hypothèses nulles vraies": ", ".join(tn),
                          **{k: v for k, v in am.procedure_table(SIM_M[sm], SIM_G[sg], true_null=tn, seed=SEED + 70 + i).items()
                             if k.startswith("erreur_globale")}}
                         for i, (name, sm, sg, tn) in enumerate(NULLS)]).set_index("configuration nulle")
TAB_FWER.to_csv(RES_DIR / "erreur_globale.csv")
print(f"Risque global d'erreur de la procédure complète (cible ≤ {am.ALPHA}) :")
display(TAB_FWER.round(3))

def rate(x):
    k = int(np.sum(x)); return k / len(x), *am.mc_ci(k, len(x))
TYPE1 = {"C1a sous b = 0": rate(SIM_M["b = 0"]["p_C1a"] <= am.ALPHA), "C1b sous b = 0,5": rate(SIM_M["b = 0,5"]["p_C1b"] <= am.ALPHA),
         "C2 sous b = 0,5": rate(SIM_M["b = 0,5"]["p_C2"] <= am.ALPHA), "C4 sous γ = 0": rate(SIM_G["γ = 0"]["p_C4"] <= am.ALPHA)}
TAB_T1 = pd.DataFrame(TYPE1, index=["taux", "IC_bas", "IC_haut"]).T
print("Erreur de type I de chaque test seul (IC exact de Monte Carlo) :")
display(TAB_T1.round(3))
TAB_G = pd.DataFrame({k: {"C4 (p ≤ 2,5 %)": float(np.mean(v["p_C4"] <= am.ALPHA)), "γ estimé médian": float(v["gamma_est"].median()),
                          "événements médians": float(v["evenements"].median())} for k, v in SIM_G.items()}).T
display(TAB_G.round(3))

Test verrouillé : 96 hôpitaux ; patients d'évaluation médiane 70 [30 ; 149] ; présents à 48 h médiane 40 ; décès attendus par hôpital ≈ 6.5 ; v_h médiane ≈ 0.169
Lactate (développement) : γ = 0.928 [0.516 ; 1.346] ; τ² résiduel (γ = 1) = 0.395
  test exact, score efficace (τ² = 0.395) : p = 0.0005 ; score non pondéré : p = 0.0180
  puissance de C4 extrapolée du développement : 0.88 à 2,5 % ; 0.81 à 1,25 % (z dév. 3.29 → z verrouillé attendu 3.12)
Simulations de C4 : 83 s
Simulations de mortalité : 586 s (500 réplications par scénario)
Puissance de C1a si C4 n'est pas rejeté (seuil 1,25 %), b du dév. : 0.926


,b,tau2,C1_gele,C3_gele,C1a_confirme,C4_confirme,C1b_confirme,C2_confirme,C1a_seul,C1b_seul,C2_seul,C4_seul,b_estime_median,largeur_IC_mediane
scénario,,,,,,,,,,,,,,
"b = 0,8",0.800,0.081,0.188,0.134,0.892,0.984,0.184,0.164,0.892,0.186,0.248,0.992,0.770,0.930
b = 1,1.000,0.081,0.420,0.214,0.966,0.982,0.522,0.492,0.966,0.524,0.602,0.992,0.961,0.897
b du dév.,0.916,0.081,0.346,0.156,0.956,0.984,0.360,0.334,0.956,0.362,0.434,0.992,0.877,0.914
"b = 0,8 ; τ² = 0,12",0.800,0.120,0.158,0.072,0.792,0.998,0.160,0.142,0.792,0.160,0.190,0.992,0.741,1.017
"b = 1 ; τ² = 0,12",1.000,0.120,0.366,0.212,0.950,0.994,0.402,0.386,0.950,0.406,0.486,0.992,0.938,0.975
"b du dév. ; τ² = 0,12",0.916,0.120,0.322,0.174,0.916,0.990,0.334,0.310,0.916,0.336,0.410,0.992,0.877,0.986
"b du dév. ; γ = 0,5",0.916,0.081,0.346,0.156,0.942,0.644,0.330,0.302,0.956,0.362,0.434,0.754,0.877,0.914


Risque global d'erreur de la procédure complète (cible ≤ 0.025) :


,hypothèses nulles vraies,erreur_globale,erreur_globale_IC_bas,erreur_globale_IC_haut
configuration nulle,,,,
"nul global : b = 0, γ = 0","C1a, C1b, C2, C4",0.032,0.018,0.051
b = 0 ; γ du dév.,"C1a, C1b, C2",0.032,0.018,0.051
"b = 0,5 ; γ du dév.","C1b, C2",0.020,0.010,0.036
b du dév. ; γ = 0,C4,0.022,0.011,0.039


Erreur de type I de chaque test seul (IC exact de Monte Carlo) :


,taux,IC_bas,IC_haut
C1a sous b = 0,0.032,0.018,0.051
"C1b sous b = 0,5",0.020,0.010,0.036
"C2 sous b = 0,5",0.024,0.012,0.042
C4 sous γ = 0,0.030,0.017,0.049


,"C4 (p ≤ 2,5 %)",γ estimé médian,événements médians
γ = 0,0.030,0.008,104.0
"γ = 0,5",0.754,0.503,108.0
γ du dév.,0.992,0.936,115.0


## Étape 5 — Amendement 1, déclaration d'accès et texte de dépôt

`amendement_1.json` cite l'empreinte complète de `specification_gelee.json`, qui reste inchangé. `depot_osf.md` est le texte du dépôt, en anglais.
Il faut déposer les deux fichiers JSON et ce texte **avant** le notebook 07, une fois la version « validée » produite.

In [13]:
# ============================================================
# ÉTAPE 5 — AMENDEMENT 1, DÉCLARATION D'ACCÈS, DÉPÔT
# ============================================================
crit_old = {c["code"]: c for c in SPEC["criteres_confirmatoires"]}
p_main = TAB_POW.loc["b du dév."]
PW_GEL, PW_C3 = float(p_main["C1_gele"]), float(p_main["C3_gele"])
BRUIT_FR = (f"variance d'échantillonnage médiane de a_h ≈ {V_MED:.3f} (environ {EXP_DEATHS:.0f} décès attendus par hôpital verrouillé) "
            f"contre τ² ≈ {TAU2_DEV:.3f} entre hôpitaux de développement")
BRUIT_EN = (f"median sampling variance of a_h ≈ {V_MED:.3f} (about {EXP_DEATHS:.0f} expected deaths per locked hospital) versus "
            f"between-hospital τ² ≈ {TAU2_DEV:.3f} in development")
if PW_GEL < 0.80:
    MOTIF_FR = (f"Puissance insuffisante du C1 gelé : {PW_GEL:.2f} sous les estimations du développement ({BRUIT_FR}). L'amendement précise "
                "aussi l'estimand, la portée de b > 1/2, les comparateurs et le contrôle de la multiplicité.")
    MOTIF_EN = (f"The frozen primary criterion had insufficient power ({PW_GEL:.2f} under the development estimates; {BRUIT_EN}). The amendment "
                "also defines the estimand, the scope of b > 1/2, the comparators and multiplicity control.")
else:
    MOTIF_FR = (f"Le C1 gelé a une puissance de {PW_GEL:.2f} ({BRUIT_FR}), mais il combine deux estimations sans estimand défini, sans "
                "comparateurs explicites ni contrôle de la multiplicité ; l'amendement les précise.")
    MOTIF_EN = (f"The frozen primary criterion had a power of {PW_GEL:.2f} ({BRUIT_EN}) but combined two estimates without a defined estimand, "
                "explicit comparators or multiplicity control; the amendment specifies them.")
DECLARATION = [
    "nb01 (création du protocole, avant la conception d'OP-Cal) a affiché des statistiques groupées des labels du test verrouillé "
    "(effectifs, décès, mortalité de l'ensemble verrouillé) ; aucune statistique par hôpital verrouillé n'a été examinée.",
    "06b, 06c et la version 1 de 06d : la fonction gelée opcal_feas.site_descriptors et le contrôle 8 du 06b lisent le fichier "
    "clients/manifest.csv complet, qui contient la mortalité par hôpital de tous les hôpitaux, y compris verrouillés. Seules les colonnes "
    "n_stays et n_train des hôpitaux de développement sont utilisées ; aucune valeur d'hôpital verrouillé n'a été utilisée, affichée ni "
    "enregistrée (vérifiable dans le code : manifest.loc[D.index, 'n_stays'] et man.loc[DEV, 'n_train']).",
    "Aucun label, aucune prédiction et aucun descripteur d'hôpital verrouillé n'a été lu : stay_labels.parquet ne contient que les "
    "hôpitaux de développement (notebook 04), et les prédictions des notebooks 02 et 05 ne portent que sur eux.",
    "06d version 3 : journal d'accès complet (tableau 'acces' de ce fichier) ; manifest lu avec les seules colonnes hospitalid, role, n_val, n_test.",
]
AMEND = {
    "document": "Amendement 1 à la spécification gelée OP-Cal (06c) — version 3",
    "statut": "validé" if VALIDATION["valide"] else "proposé (à valider avant dépôt)",
    "validation": VALIDATION,
    "date": datetime.now().isoformat(timespec="seconds"),
    "protocol_hash": protocol["hash"],
    "specification": {"fichier": "results/06c_robustesse_opcal/specification_gelee.json", "sha256": SPEC_SHA, "date": SPEC["date"], "modifiee": False},
    "declaration_acces": DECLARATION,
    "motif": MOTIF_FR,
    "affirmation_principale": ("C1a : le signal de calibration des sites prédit sans label se transporte vers des hôpitaux nouveaux. "
                               "L'amélioration (C1b, C2) est une affirmation confirmatoire secondaire, de puissance limitée sur ce jeu "
                               f"({p_main['C1b_confirme']:.2f} et {p_main['C2_confirme']:.2f} sous les estimations du développement) ; "
                               "sa preuve principale vient du développement (pente croisée b, gain de log-vraisemblance)."),
    "modifications": [
        {"critere": "C1", "avant": f"{crit_old['C1']['mesure']} ; {crit_old['C1']['seuil']}",
         "apres": "remplacé par C1a (affirmation principale) et C1b ; R²_τ et gains en |CITL| rapportés comme estimations"},
        {"critere": "C2", "avant": f"{crit_old['C2']['mesure']} ; {crit_old['C2']['seuil']}",
         "apres": "gain de log-vraisemblance par patient contre E0 ; |CITL| contre E0 et E1 descriptifs"},
        {"critere": "C3", "avant": f"{crit_old['C3']['mesure']} ; {crit_old['C3']['seuil']}",
         "apres": f"descriptif (puissance {PW_C3:.2f} sous les estimations du développement ; preuve du régime 2 : 06b, F2)"},
        {"critere": "C4", "avant": f"{crit_old['C4']['mesure']} ; {crit_old['C4']['seuil']}",
         "apres": "test exact par permutation de la statistique du score ; γ et son IC (bootstrap) rapportés comme estimations"},
        {"critere": "S1", "avant": f"{crit_old['S1']['mesure']} ; {crit_old['S1']['seuil']}", "apres": "GRU-D : C1a et C1b, descriptifs"},
    ],
    "estimand_principal": "b dans a_h = alpha + b·(â_h − moyenne de â) + u_h + e_h, u_h ~ N(0, tau_r²), e_h ~ N(0, v_h)",
    "portee_b_demi": ("Sous ce modèle et à moyenne correcte, R_constante − R_OP-Cal = (2b − 1)·Var(â) : b > 1/2 ⟺ réduction attendue de "
                      "l'erreur quadratique de calibration par rapport à la meilleure constante (référence théorique : moyenne vraie des a_h "
                      "verrouillés). Cela ne garantit ni une baisse du |CITL| dans chaque hôpital, ni un gain de log-vraisemblance, ni la "
                      "supériorité sur E1 : C2 et les comparaisons à E1 sont distincts de C1b."),
    "comparateurs": {"meilleure_constante": "référence théorique de l'interprétation de b, non estimable sans labels",
                     "E0": "aucune correction du modèle final (décalage 0)",
                     "E1": "décalage constant = moyenne REML des a_h croisés des 60 hôpitaux de développement"},
    "definitions": {
        "a_h": "CITL (offset logistique, cible bornée à [0,5 ; n − 0,5]) du modèle final x_only sur les patients val + test de l'hôpital verrouillé h",
        "v_h": "1 / Σ μ(1 − μ), μ = sigmoid(lp + a_h)",
        "â_h": "prédiction de opcal_feas.SiteEB ajusté une fois sur les 60 hôpitaux de développement (44 descripteurs gelés), covariable fixe",
        "z_h": "log de la part des séjours présents à 48 h redosés en lactate après 24 h, split train de l'hôpital verrouillé",
        "population_C4": "séjours présents à 48 h, patients val + test des hôpitaux verrouillés, hyperlactatémie (non dosé = négatif, notebook 04)",
    },
    "criteres": [
        {"code": "C1a", "H1": "b > 0", "poids_initial": 0.5, "test": "REML, Knapp-Hartung sans troncature, t à k − 2 ddl ; p unilatéral"},
        {"code": "C4", "H1": "U > 0", "poids_initial": 0.5,
         "test": (f"score efficace U = Σ_h (z_h − z̄)(O_h − E_h) / (1 + E_h·τ²), τ² = {TAU2_MECH:.4f} fixé sur le développement (hétérogénéité "
                  "résiduelle après le terme mécanistique γ = 1) ; O − E et E sous logit P(Y* = 1) = lp + alpha (alpha commun, MLE) ; z permuté entre hôpitaux, "
                  "2000 permutations (graine 2026) ; p = (1 + #{U* ≥ U}) / 2001 ; exact si z est indépendant des écarts O − E sous H0")},
        {"code": "C1b", "H1": "b > 0,5", "poids_initial": 0.0, "test": "comme C1a, H0 : b ≤ 0,5"},
        {"code": "C2", "H1": "Δℓ > 0", "poids_initial": 0.0,
         "test": "Δℓ = Σ_h Σ_i [ℓ(lp_i + â_h) − ℓ(lp_i)] / Σ_h n_h ; bootstrap de 2000 hôpitaux (graine 2026) ; p = (1 + #{Δℓ* ≤ 0}) / 2001"},
    ],
    "multiplicite": ("procédure graphique (Bretz, Maurer, Brannath et Posch 2009), unilatérale à 2,5 % : poids initiaux C1a 1/2, C4 1/2 ; "
                     "arêtes C4 → C1a, C1a → C1b, C1b → C2, C2 → C4 (poids 1). Les conclusions de mortalité ne dépendent pas de C4."),
    "regles": ["aucun hôpital verrouillé exclu",
               "modèles finaux : réglages du notebook 02, entraînés sur les 60 hôpitaux de développement, prédictions moyennées sur les seeds",
               "indépendance : patients distincts entre hôpitaux ; â_h fixes ; erreur commune du modèle final absorbée par alpha",
               "le test verrouillé est évalué une seule fois ; tout écart est déclaré comme analyse exploratoire"],
    "toujours_rapportes": ["alpha, b et R²_τ avec IC", "bootstrap de b (2000 hôpitaux, tau² réestimé)", "γ et IC, et si l'IC contient 1",
                           "|CITL| et log-vraisemblance contre E0 et E1", "EM et BBSE", "GRU-D", "S_mech_fixe", "C3 (k labels) descriptif"],
    "formulation_des_conclusions": {"C1a": "le signal de calibration des sites se transporte",
                                    "C1a sans C1b": "le signal se transporte ; l'ampleur du gain reste incertaine sur ce jeu",
                                    "C1b": "amélioration confirmée par rapport à la meilleure constante, sous le modèle",
                                    "C2": "gain de vraisemblance confirmé contre l'absence de correction",
                                    "C4": "mécanisme du dosage confirmé", "échec": "rapporté tel quel"},
    "limites_simulation": ["lp tirés parmi les patients des hôpitaux de développement", "part des présents à 48 h et taux de dosage du développement",
                           "dispersion de â_h du développement", "puissance = analyse de sensibilité sous hypothèses explicites, pas une garantie"],
    "pente_developpement": TAB_DEV.reset_index().to_dict(orient="records"),
    "gain_developpement": GAIN_DEV.reset_index().to_dict(orient="records"),
    "puissance": TAB_POW.reset_index().to_dict(orient="records"),
    "risque_global": TAB_FWER.reset_index().to_dict(orient="records"),
    "erreur_type_I": TAB_T1.reset_index().rename(columns={"index": "test"}).to_dict(orient="records"),
    "puissance_C4": TAB_G.reset_index().rename(columns={"index": "scénario"}).to_dict(orient="records"),
    "puissance_C4_extrapolee_du_developpement": {**C4_EMP, "p_dev_score_efficace": C4_DEV["p_C4"], "p_dev_non_pondere": C4_DEV_NP["p_C4"],
                                                 "remarque": "la simulation (hétérogénéité gaussienne) est optimiste pour C4 ; cette extrapolation est plus réaliste"},
    "puissance_C1a_si_C4_echoue": C1A_FALLBACK,
    "code_sha256": {**MOD_SHA, "opcal_amend.py": sha256(SRC_DIR / "opcal_amend.py")},
}

# journal d'accès résumé (les fichiers de prédictions sont regroupés par dossier)
acc = pd.DataFrame(ACCESS_LOG)
acc["groupe"] = acc["fichier"].map(lambda f: str(Path(f).parent) + "/*.parquet" if "/preds/" in f.replace("\\", "/") else f)
ACCES = (acc.groupby(["groupe", "fonction"])
            .agg(lectures=("fichier", "size"),
                 hopitaux_verrouilles=("hopitaux", lambda s: len(set().union(*s) & LOCKED)),
                 colonnes_label=("colonnes_label", lambda s: ", ".join(sorted(set().union(*map(set, s))))),
                 colonnes_demandees=("colonnes_demandees", lambda s: ", ".join(sorted({c for x in s if x for c in x})) or "toutes"))
            .reset_index())
AMEND["acces"] = ACCES.to_dict(orient="records")
amend_path = RES_DIR / "amendement_1.json"
amend_path.write_text(json.dumps(AMEND, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
AMEND_SHA = sha256(amend_path)

osf = f"""# OP-Cal — confirmatory analysis plan for the eICU locked test set (frozen specification + amendment 1, v3)

**Status:** {"validated" if VALIDATION["valide"] else "PROPOSED — not yet validated"}{(" by " + VALIDATION["par"] + " on " + VALIDATION["date"]) if VALIDATION["valide"] else ""}.
**Protocol hash:** `{protocol['hash']}` ({len(DEV)} development hospitals in {len(FOLDS)} leave-hospitals-out folds; {len(LOCKED)} locked hospitals with 100–499 stays).

## Files and fingerprints
| File | SHA-256 |
|---|---|
| `specification_gelee.json` (frozen {SPEC['date']}) | `{SPEC_SHA}` |
| `amendement_1.json` ({AMEND['date']}) | `{AMEND_SHA}` |
| `opcal_feas.py` / `opcal_robust.py` / `opcal_amend.py` | `{AMEND['code_sha256']['opcal_feas.py'][:16]}…` / `{AMEND['code_sha256']['opcal_robust.py'][:16]}…` / `{AMEND['code_sha256']['opcal_amend.py'][:16]}…` |

## Method under test
OP-Cal predicts the calibration-in-the-large offset a_h of a federated model at a new hospital without any of its labels. It uses an
empirical-Bayes (REML) meta-regression on {len(FROZEN)} label-free site descriptors (observation process, case-mix, structure, score distribution),
fitted once on out-of-federation offsets of the 60 development hospitals. For labels defined by a test (hyperlactataemia, "not tested = negative"),
the theory Y* = D·Y predicts that hospitals testing more record more events than the model expects.

## Access to the locked set
No label, prediction or descriptor of a locked hospital has been read. The protocol notebook (nb01) displayed pooled statistics of the
locked set when the split was created, before OP-Cal was designed. The frozen descriptor function and one development control read the full
manifest file, which contains per-hospital mortality for all hospitals; only development-hospital sizes were used, and no locked-hospital value was
used, displayed or stored. A complete file-access log of the amendment notebook is included in `amendement_1.json` (field `acces`).

## Why the frozen primary criterion was amended (before any access to the locked set)
{MOTIF_EN} Power was simulated with the real locked-hospital sizes ({len(N_EVAL)} hospitals, median {np.median(N_EVAL):.0f} evaluation
patients) and the exact final procedure (same tests, same numbers of bootstrap and permutation draws). The few-label criterion (k = {K_CONF})
becomes descriptive (power {PW_C3:.2f}).

## Confirmatory hypotheses (one-sided α = 2.5%, graphical procedure of Bretz et al. 2009, strong family-wise error control)
* **C1a (primary claim, initial weight 1/2):** site-level calibration slope b > 0 — REML meta-regression of locked a_h on frozen predictions, Knapp–Hartung.
* **C4 (mechanism, initial weight 1/2):** exact permutation test of the efficient score U = Σ (z_h − z̄)(O_h − E_h)/(1 + E_h·τ²) for the log
  testing rate (τ² = {TAU2_MECH:.3f}, fixed from development).
* **C1b (weight received from C1a):** b > 0.5. Under the model and with a correct mean, this is a reduction of the expected squared calibration
  error versus the best constant (a theoretical reference); it does not by itself imply a log-likelihood gain or superiority over a constant
  correction estimated in development.
* **C2 (weight received from C1b):** per-patient log-likelihood gain versus no correction > 0.
* Edges: C4 → C1a, C1a → C1b, C1b → C2, C2 → C4. Mortality conclusions do not depend on C4.

Estimated power under the development estimates (b = {B_DEV:.2f}): C1a {p_main['C1a_confirme']:.2f} ({C1A_FALLBACK:.2f} if C4 is not rejected);
C1b {p_main['C1b_confirme']:.2f}; C2 {p_main['C2_confirme']:.2f}. C4: {C4_EMP['puissance_a_0.0125']:.2f}–{C4_EMP['puissance_a_0.0250']:.2f} extrapolated from the
development test (p = {C4_DEV['p_C4']:.3f}); the simulation with Gaussian heterogeneity ({p_main['C4_confirme']:.2f}) is optimistic for C4.
Family-wise error of the full procedure under the global null:
{TAB_FWER.iloc[0]['erreur_globale']:.3f} [{TAB_FWER.iloc[0]['erreur_globale_IC_bas']:.3f}; {TAB_FWER.iloc[0]['erreur_globale_IC_haut']:.3f}].
These are sensitivity analyses under explicit simulation assumptions (scores drawn from development patients), not guarantees.

## Always reported (descriptive)
α, b and R²_τ with CIs; bootstrap of b; γ with CI (and whether it contains 1); |CITL| and log-likelihood versus no correction and versus a
constant estimated in development; label-shift estimators (EM, BBSE); GRU-D; mechanism-only offset; few-label updating (k = {K_CONF}).

## Reporting rules
C1a: "the site calibration signal transports"; without C1b: "the size of the gain remains uncertain on this set". C1b: "calibration
improvement versus the best constant confirmed, under the model". Failures are reported as such. The locked set is evaluated once.
"""
(RES_DIR / "depot_osf.md").write_text(osf, encoding="utf-8")
print(f"amendement_1.json : statut « {AMEND['statut']} » ; SHA-256 {AMEND_SHA}")
print(f"depot_osf.md écrit. Spécification gelée inchangée : SHA-256 {SPEC_SHA}")

amendement_1.json : statut « validé » ; SHA-256 cf4eceb83b8c6624f7e1f37dc2a87d29395080695d40cec2f7f79e35a5dfadc7
depot_osf.md écrit. Spécification gelée inchangée : SHA-256 302a9b448a1adcf481802ce6827e95e8c9afb623c3b7cf39ac3ca9f9eead25ae


## Étape 6 — Contrôles et rapport

In [14]:
# ============================================================
# ÉTAPE 6 — CONTRÔLES ET RAPPORT
# ============================================================
CONTROLES = []
oc.check("Protocole identique (hash)", protocol["hash"] == EXPECTED_PROTOCOL_HASH, CONTROLES)
oc.check("Spécification gelée inchangée depuis le 06c (empreinte du rapport 06c)", SPEC_SHA_06C is not None and SPEC_SHA.startswith(SPEC_SHA_06C), CONTROLES)
oc.check("Modules opcal_feas et opcal_robust identiques à ceux de la spécification", MOD_SHA == SPEC["code_sha256"], CONTROLES)
abl = pd.read_csv(RES06C / "ablation.csv").set_index(["tâche", "ensemble"])
r2_ref = abl.loc[("mortalite_x_only", "S_all − exclus (gelé)"), "R2_tau"]
oc.check(f"Prédiction gelée du développement reproduite (R²_τ {ev['R2_tau']:.4f} ; 06c {r2_ref:.4f})", abs(ev["R2_tau"] - r2_ref) < 1e-6, CONTROLES)
log = pd.DataFrame(ACCESS_LOG)
both = log[log.apply(lambda r: bool(r["hopitaux"] & LOCKED) and bool(r["colonnes_label"]), axis=1)]
oc.check("Journal d'accès : aucun fichier lu ne contient à la fois une ligne d'hôpital verrouillé et une colonne de label", both.empty, CONTROLES)
man_reads = log[log["fichier"].str.replace("\\", "/").str.endswith("clients/manifest.csv")]
oc.check("Journal d'accès : manifest lu uniquement avec hospitalid, role, n_val, n_test",
         len(man_reads) > 0 and all(r == man_cols for r in man_reads["colonnes_demandees"]), CONTROLES)
oc.check("Journal d'accès : aucun fichier hospital_*.npz ouvert", not (log["fonction"] == "np.load").any(), CONTROLES)
locked_rows = log[log["hopitaux"].map(lambda s: bool(s & LOCKED))]
oc.check("Journal d'accès : seul le manifest (colonnes d'effectifs) contient des lignes d'hôpitaux verrouillés",
         locked_rows["fichier"].str.replace("\\", "/").str.endswith("clients/manifest.csv").all(), CONTROLES)
ok_t1 = all(TAB_T1.loc[k, "IC_bas"] <= am.ALPHA for k in TAB_T1.index)
oc.check("Erreur de type I de chaque test compatible avec ≤ 2,5 % (borne basse de l'IC de Monte Carlo)", ok_t1, CONTROLES)
oc.check("Risque global de la procédure compatible avec ≤ 2,5 % sous les 4 configurations nulles",
         bool((TAB_FWER["erreur_globale_IC_bas"] <= am.ALPHA).all()), CONTROLES)
oc.check("Amendement lié à l'empreinte complète de la spécification", json.loads(amend_path.read_text(encoding="utf-8"))["specification"]["sha256"] == SPEC_SHA, CONTROLES)
CONTROLES = pd.DataFrame(CONTROLES)
print(f"\n{int(CONTROLES['ok'].sum())}/{len(CONTROLES)} contrôles réussis")
display(ACCES)

def md_table(df, digits=3):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{v:.{digits}f}")
    esc = lambda x: str(x).replace("|", "\\|")
    head = "| " + " | ".join([esc(df.index.name or "")] + [esc(c) for c in df.columns]) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    body = ["| " + " | ".join([esc(i)] + [esc(v) for v in row]) + " |" for i, row in zip(df.index, df.to_numpy())]
    return "\n".join([head, sep] + body)

rapport = f"""# Rapport 06d (version 3) — puissance du test verrouillé et amendement 1

Généré le {datetime.now():%Y-%m-%d %H:%M} par `notebooks/06d_amendement_puissance.ipynb`. Protocole `{protocol['hash']}`. Agrégats uniquement.

**Amendement 1 : statut « {AMEND['statut']} »** — SHA-256 `{AMEND_SHA}`.
Spécification gelée inchangée — SHA-256 `{SPEC_SHA}`.

**Motif :** {MOTIF_FR}

## Pente de calibration des sites (développement, {len(FROZEN)} descripteurs gelés)

{md_table(TAB_DEV)}

Gain de log-vraisemblance par patient (développement) :

{md_table(GAIN_DEV, 5)}

Lactate (développement) : γ = {G_DEV['gamma']:.3f} [{G_DEV['IC_bas']:.3f} ; {G_DEV['IC_haut']:.3f}] ; τ² résiduel = {TAU2_MECH:.3f} ;
test exact, score efficace : p = {C4_DEV['p_C4']:.4f} (non pondéré : {C4_DEV_NP['p_C4']:.4f}).

**Puissance de C4 extrapolée du développement** (plus réaliste que la simulation, qui suppose une hétérogénéité gaussienne) :
{C4_EMP['puissance_a_0.0250']:.2f} à 2,5 % et {C4_EMP['puissance_a_0.0125']:.2f} à 1,25 %.
**Puissance de C1a si C4 n'est pas rejeté** (seuil 1,25 %) : {C1A_FALLBACK:.2f}. Les conclusions de mortalité ne dépendent pas de C4.

## Puissance ({len(N_EVAL)} hôpitaux verrouillés, médiane {np.median(N_EVAL):.0f} patients d'évaluation ; procédure graphique)

Var(â) = {VAR_AHAT:.4f} ; τ² du développement = {TAU2_DEV:.4f} ; v_h médiane ≈ {V_MED:.3f} ; k = {K_CONF} pour C3 ; {N_REP_M} réplications par scénario ({N_REP_G} pour C4).
Les colonnes « _confirme » correspondent à la procédure complète ; « _seul » au test isolé à 2,5 %.

{md_table(TAB_POW[pow_cols])}

## Risque global d'erreur de la procédure complète (cible ≤ 0,025)

{md_table(TAB_FWER)}

Erreur de type I de chaque test seul :

{md_table(TAB_T1)}

C4 (lactate) :

{md_table(TAB_G)}

## Journal d'accès aux fichiers

{md_table(ACCES.set_index('groupe'))}

## Contrôles

{md_table(CONTROLES.set_index('controle'))}
"""
(RES_DIR / "rapport_06d.md").write_text(rapport, encoding="utf-8")
print(f"Écrit : {RES_DIR / 'rapport_06d.md'} | amendement_1.json | depot_osf.md | puissance.csv | erreur_globale.csv | pente_developpement.csv")

OK     Protocole identique (hash)
OK     Spécification gelée inchangée depuis le 06c (empreinte du rapport 06c)
OK     Modules opcal_feas et opcal_robust identiques à ceux de la spécification
OK     Prédiction gelée du développement reproduite (R²_τ 0.6077 ; 06c 0.6077)
OK     Journal d'accès : aucun fichier lu ne contient à la fois une ligne d'hôpital verrouillé et une colonne de label
OK     Journal d'accès : manifest lu uniquement avec hospitalid, role, n_val, n_test
OK     Journal d'accès : aucun fichier hospital_*.npz ouvert
OK     Journal d'accès : seul le manifest (colonnes d'effectifs) contient des lignes d'hôpitaux verrouillés
OK     Erreur de type I de chaque test compatible avec ≤ 2,5 % (borne basse de l'IC de Monte Carlo)
OK     Risque global de la procédure compatible avec ≤ 2,5 % sous les 4 configurations nulles
OK     Amendement lié à l'empreinte complète de la spécification

11/11 contrôles réussis


,groupe,fonction,lectures,hopitaux_verrouilles,colonnes_label,colonnes_demandees
0,..\data_v1\clients\manifest.csv,read_csv,1,96,,"hospitalid, n_test, n_val, role"
1,..\data_v1\splits\protocol.json,read_text,1,0,,toutes
2,..\results\02_baselines\preds/*.parquet,read_parquet,15,0,y,toutes
3,..\results\04_observation_label\stay_labels.pa...,read_parquet,1,0,"aki_cc, aki_neg, cr_peak, hyper_cc, hyper_neg,...",toutes
4,..\results\05_raccourci_masquage\preds/*.parquet,read_parquet,25,0,,toutes
5,..\results\06b_faisabilite_opcal\descripteurs_...,read_csv,1,0,,toutes
6,..\results\06c_robustesse_opcal\rapport_06c.md,read_text,1,0,,toutes
7,..\results\06c_robustesse_opcal\specification_...,read_text,1,0,,toutes


Écrit : ..\results\06d_amendement_puissance\rapport_06d.md | amendement_1.json | depot_osf.md | puissance.csv | erreur_globale.csv | pente_developpement.csv
